# 07 · Comparing Language Models

**Goal.** Decide which language model ("backbone") best represents our text: **MuRIL vs XLM-R vs IndicBERT**.

**Why it matters.** The backbone is the foundation of the whole model, so we pick it with evidence, not
assumption. We also check whether using longer text (128 vs 256 tokens) helps.

**What happens below.** Train a simple classifier on top of each frozen backbone and compare their scores
on the held-out test set.

**Output:** `data/stage2_semantic/backbone_benchmark.json`.

## 0. Environment & dependencies

In [1]:
import os, re, sys, json, gc, time, logging, warnings
from pathlib import Path

for _stream in ("stdout", "stderr"):
    try:
        getattr(sys, _stream).reconfigure(encoding="utf-8")
    except Exception:
        pass

import numpy as np

def _try(name):
    try:
        __import__(name); return True
    except Exception:
        return False

HAS_TORCH = _try("torch") and _try("transformers")
HAS_SKLEARN = _try("sklearn")

warnings.filterwarnings("ignore")
logging.basicConfig(level=logging.INFO, format="%(asctime)s | %(levelname)s | %(message)s")
log = logging.getLogger("bench")
log.info("deps -> torch+transformers=%s sklearn=%s", HAS_TORCH, HAS_SKLEARN)
assert HAS_TORCH and HAS_SKLEARN, "notebook 07 needs torch + transformers + sklearn (project venv)."
PART_VERSION = "1.0"
SEED = 42
np.random.seed(SEED)

hinglish-mental-health-detection\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


2026-08-07 21:43:42,928 | INFO | deps -> torch+transformers=True sklearn=True


## 1. Configuration

The three backbones and two token caps. IndicBERT/XLM-R download ~1–2 GB each on first run;
per-backbone load is wrapped so a failure is recorded and skipped rather than crashing the benchmark.

In [2]:
import torch
torch.manual_seed(SEED)

def find_project_root():
    here = Path.cwd().resolve()
    for d in [here, *here.parents]:
        if (d / "data" / "stage2_semantic").exists() or (d / ".env").exists():
            return d
    return here

PROJECT_ROOT = find_project_root()
SEM_DIR = PROJECT_ROOT / "data" / "stage2_semantic"
OUTPUT_DIR = SEM_DIR

BACKBONES = [
    ("muril",     "google/muril-base-cased"),
    ("indicbert", "ai4bharat/indic-bert"),
    ("xlmr",      "xlm-roberta-base"),
]
TOKEN_CAPS = [128, 256]
BATCH = 16
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
TIER_NAMES = {0: "Neutral", 1: "Stress", 2: "Moderate", 3: "Crisis"}

log.info("device=%s | backbones=%s | token caps=%s", DEVICE, [b[0] for b in BACKBONES], TOKEN_CAPS)

2026-08-07 21:43:42,948 | INFO | device=cpu | backbones=['muril', 'indicbert', 'xlmr'] | token caps=[128, 256]


## 2. Load the preprocessed corpus (TRAIN = youtube+synthetic, TEST = reddit+twitter frozen)

In [3]:
def load_jsonl(path):
    rows = []
    with open(path, encoding="utf-8") as fh:
        for line in fh:
            line = line.strip()
            if line:
                rows.append(json.loads(line))
    return rows

train_rows = load_jsonl(SEM_DIR / "corpus_preprocessed_train.jsonl")
test_rows  = load_jsonl(SEM_DIR / "corpus_preprocessed_test.jsonl")

def texts_of(rows):
    return [str(r.get("text_muril") or r.get("post_text_clean") or "") for r in rows]

X_train_txt, y_train = texts_of(train_rows), np.array([int(r["tier"]) for r in train_rows])
X_test_txt,  y_test  = texts_of(test_rows),  np.array([int(r["tier"]) for r in test_rows])

# P3 leakage guard
assert set(r["platform"] for r in train_rows) <= {"youtube", "synthetic"}, "TRAIN platform leak!"
assert set(r["platform"] for r in test_rows)  <= {"reddit", "twitter"},   "TEST platform leak!"
log.info("train=%d (tiers %s) | test=%d (tiers %s)",
         len(train_rows), {int(t): int((y_train == t).sum()) for t in range(4)},
         len(test_rows),  {int(t): int((y_test == t).sum()) for t in range(4)})

2026-08-07 21:43:43,224 | INFO | train=1971 (tiers {0: 1545, 1: 82, 2: 255, 3: 89}) | test=2768 (tiers {0: 1581, 1: 545, 2: 596, 3: 46})


## 3. Frozen-encoder mean-pool embedding (shared)

In [4]:
from transformers import AutoTokenizer, AutoModel

@torch.no_grad()
def embed(model, tok, texts, max_tokens, batch=BATCH):
    out = []
    for i in range(0, len(texts), batch):
        chunk = [t if t.strip() else "[UNK]" for t in texts[i:i + batch]]
        enc = tok(chunk, padding=True, truncation=True, max_length=max_tokens, return_tensors="pt").to(DEVICE)
        hs = model(**enc).last_hidden_state
        mask = enc["attention_mask"].unsqueeze(-1).float()
        summed = (hs * mask).sum(1); cnt = mask.sum(1).clamp(min=1e-9)
        out.append((summed / cnt).cpu().numpy())
    return np.vstack(out)

def load_backbone(model_id):
    # robust load: try fast tokenizer, fall back to slow (IndicBERT = sentencepiece/ALBERT)
    try:
        tok = AutoTokenizer.from_pretrained(model_id)
    except Exception:
        tok = AutoTokenizer.from_pretrained(model_id, use_fast=False)
    model = AutoModel.from_pretrained(model_id).to(DEVICE).eval()
    return model, tok

log.info("embedding helpers ready")

2026-08-07 21:43:46,373 | INFO | embedding helpers ready


## 4. Linear-probe metrics (TRAIN→TEST)

StandardScaler here is legitimate — it aids the **classifier's** optimisation (P4 reserves cosine for the
*similarity* claim, not the downstream probe). Logistic regression with balanced class weights; report
accuracy, macro-F1, per-tier F1, **Tier-3 (Crisis) recall**, and MCC on the frozen TEST.

In [5]:
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import f1_score, accuracy_score, recall_score, matthews_corrcoef

def probe_eval(Xtr, ytr, Xte, yte):
    sc = StandardScaler().fit(Xtr)
    clf = LogisticRegression(max_iter=2000, class_weight="balanced", C=1.0)
    clf.fit(sc.transform(Xtr), ytr)
    pred = clf.predict(sc.transform(Xte))
    per_tier_f1 = f1_score(yte, pred, labels=[0, 1, 2, 3], average=None, zero_division=0)
    return {
        "accuracy": round(float(accuracy_score(yte, pred)), 4),
        "macro_f1": round(float(f1_score(yte, pred, average="macro", zero_division=0)), 4),
        "mcc": round(float(matthews_corrcoef(yte, pred)), 4),
        "per_tier_f1": {TIER_NAMES[i]: round(float(per_tier_f1[i]), 4) for i in range(4)},
        "tier3_recall": round(float(recall_score(yte, pred, labels=[3], average="macro", zero_division=0)), 4),
    }
log.info("probe ready")

2026-08-07 21:43:46,444 | INFO | probe ready


## 5. Run the benchmark grid (3 backbones × 2 token caps)

Each backbone is loaded once, embeds TRAIN+TEST at both caps, then is freed (CPU memory). Per-backbone load
failures are recorded and skipped so the rest of the grid still completes.

In [6]:
results = {}
load_errors = {}
for name, model_id in BACKBONES:
    t0 = time.time()
    try:
        log.info("loading %s (%s) ...", name, model_id)
        model, tok = load_backbone(model_id)
    except Exception as e:
        load_errors[name] = f"{type(e).__name__}: {e}"
        log.warning("SKIP %s — load failed: %s", name, e)
        continue
    results[name] = {"model_id": model_id, "by_token_cap": {}}
    for cap in TOKEN_CAPS:
        te0 = time.time()
        Xtr = embed(model, tok, X_train_txt, cap)
        Xte = embed(model, tok, X_test_txt, cap)
        m = probe_eval(Xtr, y_train, Xte, y_test)
        m["embed_seconds"] = round(time.time() - te0, 1)
        m["hidden_dim"] = int(Xtr.shape[1])
        results[name]["by_token_cap"][str(cap)] = m
        log.info("  %s @%d -> acc=%.3f macroF1=%.3f mcc=%.3f T3rec=%.3f (%.0fs)",
                 name, cap, m["accuracy"], m["macro_f1"], m["mcc"], m["tier3_recall"], m["embed_seconds"])
    del model, tok; gc.collect()
    log.info("%s done in %.0fs", name, time.time() - t0)

print("load errors:", load_errors if load_errors else "none")

2026-08-07 21:43:46,456 | INFO | loading muril (google/muril-base-cased) ...


2026-08-07 21:43:46,753 | INFO | HTTP Request: HEAD https://huggingface.co/google/muril-base-cased/resolve/main/config.json "HTTP/1.1 307 Temporary Redirect"


2026-08-07 21:43:46,754 | WARNING | Warning: You are sending unauthenticated requests to the HF Hub. Please set a HF_TOKEN to enable higher rate limits and faster downloads.


2026-08-07 21:43:46,766 | INFO | HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/google/muril-base-cased/afd9f36c7923d54e97903922ff1b260d091d202f/config.json "HTTP/1.1 200 OK"


2026-08-07 21:43:47,006 | INFO | HTTP Request: HEAD https://huggingface.co/google/muril-base-cased/resolve/main/tokenizer_config.json "HTTP/1.1 307 Temporary Redirect"


2026-08-07 21:43:47,018 | INFO | HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/google/muril-base-cased/afd9f36c7923d54e97903922ff1b260d091d202f/tokenizer_config.json "HTTP/1.1 200 OK"


... (37 outputs left out)

2026-08-07 22:47:14,641 | INFO |   xlmr @128 -> acc=0.565 macroF1=0.298 mcc=0.145 T3rec=0.065 (1239s)


2026-08-07 23:08:10,600 | INFO |   xlmr @256 -> acc=0.573 macroF1=0.301 mcc=0.160 T3rec=0.043 (1256s)


2026-08-07 23:08:11,186 | INFO | xlmr done in 2500s


load errors: {'indicbert': 'OSError: You are trying to access a gated repo.\nMake sure to have access to it at https://huggingface.co/ai4bharat/indic-bert.\n401 Client Error. (Request ID: Root=1-6a760e3f-1fab1f62403be4072c9a5e0d;0822977f-d74b-4ae4-97bb-871e472f23a4)\n\nCannot access gated repo for url https://huggingface.co/ai4bharat/indic-bert/resolve/main/config.json.\nAccess to model ai4bharat/indic-bert is restricted. You must have access to it and be authenticated to access it. Please log in.'}


## 6. Backbone comparison + token sensitivity tables

In [7]:
# pick the primary token cap (128) for the headline backbone comparison
PRIMARY_CAP = "128"
print("=== BACKBONE BENCHMARK (frozen-encoder linear probe, @128 tokens, frozen TEST) ===")
print(f"{'backbone':<12}{'acc':>8}{'macroF1':>9}{'mcc':>8}{'T3rec':>8}")
ranked = []
for name in results:
    m = results[name]["by_token_cap"].get(PRIMARY_CAP)
    if m:
        print(f"{name:<12}{m['accuracy']:>8.3f}{m['macro_f1']:>9.3f}{m['mcc']:>8.3f}{m['tier3_recall']:>8.3f}")
        ranked.append((name, m["macro_f1"]))
best_backbone = max(ranked, key=lambda x: x[1])[0] if ranked else None
print("\nbest backbone by macro-F1 @128:", best_backbone)

token_sensitivity = {}
print("\n=== TOKEN SENSITIVITY (macro-F1: 128 vs 256) ===")
for name in results:
    caps = results[name]["by_token_cap"]
    if "128" in caps and "256" in caps:
        d = round(caps["256"]["macro_f1"] - caps["128"]["macro_f1"], 4)
        token_sensitivity[name] = {"f1_128": caps["128"]["macro_f1"], "f1_256": caps["256"]["macro_f1"], "delta_256_minus_128": d}
        print(f"  {name:<12} 128={caps['128']['macro_f1']:.3f}  256={caps['256']['macro_f1']:.3f}  delta={d:+.4f}")

=== BACKBONE BENCHMARK (frozen-encoder linear probe, @128 tokens, frozen TEST) ===
backbone         acc  macroF1     mcc   T3rec
muril          0.602    0.332   0.235   0.065
xlmr           0.565    0.298   0.145   0.065

best backbone by macro-F1 @128: muril

=== TOKEN SENSITIVITY (macro-F1: 128 vs 256) ===
  muril        128=0.332  256=0.345  delta=+0.0131
  xlmr         128=0.298  256=0.301  delta=+0.0039


## 7. Persist reports

In [8]:
benchmark = {
    "part": "stage2_part2.3_backbone_benchmark",
    "version": PART_VERSION,
    "principle": "P1; P3 (train=youtube+synthetic, test=reddit+twitter FROZEN);  backbone;  token cap",
    "intensity": "D5 frozen-encoder + linear-probe (indicative); full multi-seed fine-tune grid = scale-up",
    "device": DEVICE, "token_caps": TOKEN_CAPS,
    "train_n": len(train_rows), "test_n": len(test_rows),
    "train_tier_dist": {int(t): int((y_train == t).sum()) for t in range(4)},
    "test_tier_dist": {int(t): int((y_test == t).sum()) for t in range(4)},
    "results": results,
    "load_errors": load_errors,
    "best_backbone_macro_f1_at128": best_backbone,
    "honest_note": "Frozen-encoder probe is NOT a full fine-tune; ranks the backbones' OUT-OF-THE-BOX "
                   "representations on this pilot corpus. Tier-3 test support is tiny -> Tier-3 recall is "
                   "noisy. Indicative, not load-bearing; full fine-tune grid + >=5 seeds is the scale-up.",
}
(OUTPUT_DIR / "backbone_benchmark.json").write_text(json.dumps(benchmark, indent=2, ensure_ascii=False), encoding="utf-8")
(OUTPUT_DIR / "token_sensitivity.json").write_text(json.dumps(
    {"metric": "macro_f1", "by_backbone": token_sensitivity,
     "note": "128 vs 256 differs only for posts >128 tokens; a small delta justifies the 128 cap for this "
             "comment-dominant corpus."}, indent=2, ensure_ascii=False), encoding="utf-8")
log.info("wrote backbone_benchmark.json + token_sensitivity.json")

2026-08-07 23:08:11,213 | INFO | wrote backbone_benchmark.json + token_sensitivity.json


## 8. Benchmark card & This step is complete when

**notebook 07 is complete when:** the 3-backbone × 2-token-cap frozen-probe grid ran (or recorded a load error),
each cell reporting accuracy / macro-F1 / MCC / per-tier F1 / Tier-3 recall on the **frozen TEST**; the
backbone comparison + token-sensitivity tables are written; and the indicative-vs-scale-up caveat is recorded.
**No human.**

In [9]:
print("=" * 64); print("Objective 2 · notebook 07 — BACKBONE BENCHMARK CARD"); print("=" * 64)
print("backbones run :", list(results.keys()), "| load errors:", list(load_errors.keys()) or "none")
print("best @128 (macro-F1):", best_backbone)
for name in results:
    m = results[name]["by_token_cap"].get("128")
    if m:
        print(f"  {name:<12} acc={m['accuracy']:.3f} macroF1={m['macro_f1']:.3f} mcc={m['mcc']:.3f} "
              f"T3rec={m['tier3_recall']:.3f} perTierF1={m['per_tier_f1']}")
dod = all([
    (OUTPUT_DIR / "backbone_benchmark.json").exists(),
    (OUTPUT_DIR / "token_sensitivity.json").exists(),
    len(results) >= 1,
])
print("\nCompletion check:", "PASS" if dod else "INCOMPLETE")

Objective 2 · notebook 07 — BACKBONE BENCHMARK CARD
backbones run : ['muril', 'xlmr'] | load errors: ['indicbert']
best @128 (macro-F1): muril
  muril        acc=0.602 macroF1=0.332 mcc=0.235 T3rec=0.065 perTierF1={'Neutral': 0.7437, 'Stress': 0.0942, 'Moderate': 0.4221, 'Crisis': 0.069}
  xlmr         acc=0.565 macroF1=0.298 mcc=0.145 T3rec=0.065 perTierF1={'Neutral': 0.7143, 'Stress': 0.0847, 'Moderate': 0.3269, 'Crisis': 0.0645}

Completion check: PASS
